## Customer Review Handling 

In [81]:
from langgraph.graph import StateGraph, START, END
from langchain_community.chat_models import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field, ValidationError
from typing import Literal
import json
from dotenv import load_dotenv
import os

load_dotenv()

True

In [82]:
model = ChatGoogleGenerativeAI(api_key=os.getenv("GOOGLE_API_KEY"), model="gemini-2.5-flash", temperature=0.2)

# Structure of sentiment analysis state (format)
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field( description="Sentiment of the review" )


class DiagnosticSchema(BaseModel):
    issue_type: Literal["network", "UI/UX", "software", "other"] = Field( description="Type of the issue" )
    urgency: Literal["low", "medium", "high"] = Field( description="Urgency level of the issue" )
    tone: Literal["angry", "frustrated", "confuse"] = Field( description="Tone of the response" )

structured_model = model.with_structured_output(SentimentSchema)
diagnostic_model = model.with_structured_output(DiagnosticSchema)


### State of Graph

In [83]:
class ReviewState(BaseModel):
    customer_name: str = Field( default=None, description="Name of the customer" )
    customer_email: str = Field( default=None, description="Email of the customer" )
    review: str = Field( description="Customer review text" )
    sentiment: str = Field( default=None, description="Sentiment of the review" )
    diagnosis: str = Field( default=None, description="Diagnostic information based on the review" )
    response: str = Field( default=None, description="Response to the customer based on sentiment" )

### Functions of Nodes

In [84]:

# Find sentiment of review 

def analyze_sentiment(state: ReviewState) -> dict:

    # prompt to analyze sentiment
    prompt = f"""Analyze the sentiment of the following customer review and classify it as positive, negative, or neutral.
    Review: "{state.review}"
    """
    result = structured_model.invoke(prompt)
    return {"sentiment": result.sentiment}


def check_sentiment(state: ReviewState) -> Literal["positive_response", "negative_response", "run_diagnosis"]:
    if state.sentiment == "positive" :
        return "positive_response"
    else:
        return "run_diagnosis"

def positive_response(state: ReviewState) -> dict:

    # prompt to generate positive response
    prompt = f"""Generate a positive response to the following customer review:
    Review: {state.review}
    customer name: {state.customer_name}
    customer email: {state.customer_email}
    """
    result = model.invoke(prompt)
    return {"response": result.content}

def run_diagnosis(state:ReviewState) -> dict:

    # prompt to diagnose issue
    prompt = f"""The following customer review indicates a negative sentiment. Diagnose the issue by classifying it into one of the following categories: network, UI/UX, software, or other. Also, determine the urgency level (low, medium, high) and the tone of the response (angry, frustrated, confused).
    Review: "{state.review}"

    return the diagnosis in the format: dict with keys 'issue_type', 'urgency', and 'tone'.
    """
    result = diagnostic_model.invoke(prompt)
    diagnosis = f"Issue Type: {result.issue_type}, Urgency: {result.urgency}, Tone: {result.tone}"
    return {"diagnosis": diagnosis}

def negative_response(state: ReviewState) -> dict:

    # prompt to generate negative response
    prompt = f"""Generate a compassionate and helpful response to the following customer review, taking into account the diagnosed issue:
    Review: {state.review}
    customer name: {state.customer_name}
    customer email: {state.customer_email}
    Diagnosis: {state.diagnosis}
    """
    result = model.invoke(prompt)
    return {"response": result.content}




### Graph Structuring 

In [85]:
graph  = StateGraph(ReviewState)


# add nodes to graph

graph.add_node('find_sentiment', analyze_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)


# add edges to graph 

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [86]:
initial_state = {
    "review": "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.",
    "customer_name": "John Doe",
    "customer_email": "john.doe@example.com"
}

final_state = workflow.invoke(initial_state)
final_state

{'customer_name': 'John Doe',
 'customer_email': 'john.doe@example.com',
 'review': 'I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.',
 'sentiment': 'negative',
 'diagnosis': 'Issue Type: software, Urgency: high, Tone: frustrated',
 'response': "Subject: We're So Sorry You're Having Trouble Logging In, John - We're On It!\n\nDear John Doe,\n\nPlease accept our sincerest apologies for the incredibly frustrating experience you've had trying to log in to our app for over an hour. We completely understand how unacceptable it is when basic functionality like logging in is affected, especially when the app keeps freezing on the authentication screen.\n\nThank you for letting us know you've already tried reinstalling the app; that helps us understand the persistence of this issue. We recognize that this is a high